# Mecha-nudges for Machines — Walkthrough

This notebook walks through the key ideas and code behind the paper [Mecha-nudges for Machines](https://arxiv.org/abs/2603.23433). We cover:

1. **The measurement problem**: How do you quantify whether text is optimized for AI agents?
2. **Constrained generation**: Forcing an LLM to make a binary decision with calibrated probabilities
3. **Class balancing**: Undersampling to ensure fair model training
4. **Fine-tuning**: Training the classifiers used to compute $\mathcal{V}$-Information
5. **$\mathcal{V}$-Information**: Computing how much product text reduces an LLM's decision uncertainty
6. **Main result**: Reproducing the paper's regression on the included data

## 1. The Measurement Problem

After ChatGPT's release (November 30, 2022), sellers on platforms like Etsy may have started optimizing their product listings for AI agents — not just humans. But how do you measure this?

We use **Pointwise $\mathcal{V}$-Information (PVI)** from [Xu et al. (2020)](https://arxiv.org/abs/2003.12298):

$$\text{PVI}(x \to y) = -\log_2 P(y \mid \emptyset) + \log_2 P(y \mid x)$$

- $P(y \mid \emptyset)$: The model's predicted probability of the gold label $y$ when it sees **no text** (baseline)
- $P(y \mid x)$: The model's predicted probability of the gold label $y$ when it sees **the actual product text**
- PVI is evaluated at each listing's gold label $y$, so it depends on both the text and the label

If post-ChatGPT listings have higher PVI than pre-ChatGPT listings, the text has become more *machine-readable* — consistent with mecha-nudging.

## 2. Constrained Generation (Label Creation)

The first pipeline step prompts an LLM to make a binary SELECT/PASS decision for each product listing. Rather than parsing free-text, we use a custom `LogitsProcessor` that constrains output to exactly one token.

### The prompt

We frame the LLM as a selective Etsy browsing assistant (prompt V4):

In [ ]:
with open("prompts/prompt_v4.txt") as f:
    prompt_template = f.read()

print(prompt_template)

The prompt uses three placeholders: `{product_info}`, `{positive_token}`, `{negative_token}`. Here's how a prompt is constructed for a specific product:

In [ ]:
import sys
sys.path.insert(0, "src")
from generation_utils import prompt_creator

# Example product
product = {
    "Title": "Handmade Ceramic Mug - Speckled Blue Glaze",
    "Item_Details": "Wheel-thrown stoneware mug with a unique speckled blue glaze. Holds 12oz. Dishwasher and microwave safe."
}

full_prompt, product_info = prompt_creator(product, positive_token="SELECT", negative_token="PASS")
print(full_prompt)

### The constrained decoding mechanism

The key innovation is `PosNegLogitsProcessor`. On each generation step:

1. **Step 1** (first token): Mask the entire vocabulary except the two decision tokens. Extract calibrated probabilities via softmax over those two logits. Output the higher-probability token.
2. **Step 2+**: Force the EOS token to stop generation.

This guarantees every response is a valid binary decision, and gives us calibrated P(SELECT) and P(PASS).

In [ ]:
import numpy as np

# Core logic of the constrained decoding (simplified for illustration)
# In practice this runs inside HuggingFace's generate() loop via PosNegLogitsProcessor

def explain_constrained_decoding():
    """Walk through the logic of PosNegLogitsProcessor."""
    
    # Suppose the model produces these raw logits over its vocabulary
    vocab_size = 32000
    raw_logits = np.random.randn(vocab_size)
    
    # Token IDs for SELECT and PASS (hypothetical)
    select_id, pass_id = 5765, 15114
    
    # Step 1: Extract logits for just the two decision tokens
    decision_logits = np.array([raw_logits[select_id], raw_logits[pass_id]])
    print(f"Raw logits — SELECT: {decision_logits[0]:.3f}, PASS: {decision_logits[1]:.3f}")
    
    # Calibrated probabilities via softmax
    exp_logits = np.exp(decision_logits - decision_logits.max())  # numerically stable softmax
    probs = exp_logits / exp_logits.sum()
    print(f"P(SELECT) = {probs[0]:.4f}")
    print(f"P(PASS)   = {probs[1]:.4f}")
    
    # The processor masks everything else to -inf, then outputs the winner
    if probs[0] > probs[1]:
        print("→ Decision: SELECT")
    else:
        print("→ Decision: PASS")

explain_constrained_decoding()

## 4. Fine-Tuning

After labeling and balancing, we fine-tune LoRA-adapted classifiers on the balanced labels. This step requires GPU infrastructure (we used 4x H100 GPUs) and is not reproducible in this notebook, but we describe the setup here for completeness.

**Four models are trained** — a content/null pair for each time period:

| Model | Period | Training data | What it estimates |
|-------|--------|--------------|-------------------|
| Content model (pre) | Pre-ChatGPT | Actual product text | $P(Y \mid x)$ for pre-period listings |
| Null model (pre) | Pre-ChatGPT | Single space `" "` (empty input) | $P(Y \mid \emptyset)$ — pre-period base rate |
| Content model (post) | Post-ChatGPT | Actual product text | $P(Y \mid x)$ for post-period listings |
| Null model (post) | Post-ChatGPT | Single space `" "` (empty input) | $P(Y \mid \emptyset)$ — post-period base rate |

Why separate models per period? The label base rates differ across periods (~16% positive pre-ChatGPT vs. ~5% post-ChatGPT with GPT-5-mini). A single null model would learn a blended base rate that is wrong for both periods. Training per-period models ensures that each PVI score is computed relative to the correct baseline for its period.

The difference in each pair's $\mathcal{V}$-entropies is the PVI.

**LoRA configuration** (applied to Llama-3.1-8B-Instruct):

```
Rank (r):        32        Target modules:  q, k, v, o, gate, up, down projections
Alpha:           64        Task:            Sequence classification (2 labels)
Dropout:         0.05      Precision:       bfloat16
```

**Training hyperparameters:**

```
Batch size: 12              Epochs: 3               Learning rate: 2e-4
Warmup ratio: 0.03          Weight decay: 0.01      Max sequence length: 1024
Gradient accumulation: 16   Gradient clipping: 1.0
```

Each period's labeled dataset is split 80/10/10 into train/validation/test. The null model uses the same splits but with all text replaced by a single space — it learns only the label prior $P(Y)$ for that period.

See `src/finetuning.py` for the full implementation. The PVI results from our trained models are included in `data/` so you can reproduce the regression without re-training.

## 3. Fine-Tuning

After labeling all listings with SELECT/PASS decisions, we fine-tune LoRA-adapted classifiers on the generated labels. This step requires GPU infrastructure (we used 4x H100 GPUs) and is not reproducible in this notebook, but we describe the setup here for completeness.

**Four models are trained** — a content/null pair for each time period:

| Model | Period | Training data | What it estimates |
|-------|--------|--------------|-------------------|
| Content model (pre) | Pre-ChatGPT | Actual product text | $P(Y \mid x)$ for pre-period listings |
| Null model (pre) | Pre-ChatGPT | Single space `" "` (empty input) | $P(Y \mid \emptyset)$ — pre-period base rate |
| Content model (post) | Post-ChatGPT | Actual product text | $P(Y \mid x)$ for post-period listings |
| Null model (post) | Post-ChatGPT | Single space `" "` (empty input) | $P(Y \mid \emptyset)$ — post-period base rate |

Why separate models per period? The label base rates differ across periods (~16% positive pre-ChatGPT vs. ~5% post-ChatGPT with GPT-5-mini). A single null model would learn a blended base rate that is wrong for both periods. Training per-period models ensures that each PVI score is computed relative to the correct baseline for its period.

The difference in each pair's $\mathcal{V}$-entropies is the PVI.

**LoRA configuration** (applied to Llama-3.1-8B-Instruct):

```
Rank (r):        32        Target modules:  q, k, v, o, gate, up, down projections
Alpha:           64        Task:            Sequence classification (2 labels)
Dropout:         0.05      Precision:       bfloat16
```

**Training hyperparameters:**

```
Batch size: 12              Epochs: 3               Learning rate: 2e-4
Warmup ratio: 0.03          Weight decay: 0.01      Max sequence length: 1024
Gradient accumulation: 16   Gradient clipping: 1.0
```

Each period's labeled dataset is split 80/10/10 into train/validation/test. The null model uses the same splits but with all text replaced by a single space — it learns only the label prior $P(Y)$ for that period.

See `src/finetuning.py` for the full implementation. The PVI results from our trained models are included in `data/` so you can reproduce the regression without re-training.

## 5. $\mathcal{V}$-Information Computation

After fine-tuning, we run both models on the held-out test set. For each listing, we get:
- **Regular model**: $P(Y \mid x)$ — the classifier's confidence given the actual text
- **Null model**: $P(Y \mid \emptyset)$ — the classifier's confidence given nothing

The $\mathcal{V}$-entropy for each listing is:

$$H_{\mathcal{V}}(Y \mid x) = -\log_2 P_{\text{model}}(y_i \mid x_i)$$

where $y_i$ is the gold label and $P_{\text{model}}$ is the fine-tuned classifier's predicted probability.

Here is the core computation from `v_info.py`:

In [ ]:
import numpy as np

def compute_pvi(prob_given_text: float, prob_given_empty: float) -> float:
    """Compute Pointwise V-Information for a single example.
    
    Args:
        prob_given_text: P(gold_label | product_text) from regular model
        prob_given_empty: P(gold_label | empty) from null model
    
    Returns:
        PVI in bits. Positive = text is informative.
    """
    H_yx = -np.log2(prob_given_text)    # entropy given text
    H_yb = -np.log2(prob_given_empty)   # entropy given nothing (baseline)
    return H_yb - H_yx

# Example: model is very confident given text, uncertain given nothing
pvi = compute_pvi(prob_given_text=0.95, prob_given_empty=0.55)
print(f"PVI = {pvi:.3f} bits")
print(f"  H(Y|text)  = {-np.log2(0.95):.3f} bits (low uncertainty — text is helpful)")
print(f"  H(Y|empty) = {-np.log2(0.55):.3f} bits (high uncertainty — no text)")

# Example: model is uncertain either way (uninformative text)
pvi2 = compute_pvi(prob_given_text=0.55, prob_given_empty=0.55)
print(f"\nPVI = {pvi2:.3f} bits (text provides no additional signal)")

## 6. Main Result

We include the PVI results from the main experiment (SELECT/PASS tokens, balanced medium dataset, labeled with GPT-5-mini in the paper). The code defaults to Gemma-3-27B-IT for reproducibility with open-weights models. The data below is the **held-out test set** (10% split) from the class-balanced medium sample (~210k listings subsampled from 1M total across both periods). Let's load them and run the paper's baseline regression:

$$\text{PVI}_i = \beta \cdot \text{after}_i + \alpha + \varepsilon_i$$

In [ ]:
import pandas as pd

before = pd.read_json("data/before_PVI.json", lines=True)
after = pd.read_json("data/after_PVI.json", lines=True)

print(f"Pre-ChatGPT listings:  {len(before):,}")
print(f"Post-ChatGPT listings: {len(after):,}")
print(f"\nColumns: {list(before.columns)}")
before.head(3)

In [ ]:
# Tag and combine
before["time_period"] = "before"
after["time_period"] = "after"
df = pd.concat([before, after], ignore_index=True)

print(f"Mean PVI (before): {df[df.time_period == 'before']['PVI'].mean():.4f}")
print(f"Mean PVI (after):  {df[df.time_period == 'after']['PVI'].mean():.4f}")
print(f"Difference:        {df[df.time_period == 'after']['PVI'].mean() - df[df.time_period == 'before']['PVI'].mean():.4f}")

The regression above should output a coefficient on `after` of approximately **0.143** (p < 0.01), meaning post-ChatGPT listings contain 0.143 additional bits of machine-usable information on average.